# Phase 5 - XAI vs Pseudo-Concept Evaluation Pilot

This notebook performs a **pilot evaluation on 100 images** before running the full Phase 5 evaluation.

The goal is to check whether XAI saliency maps align with clinically motivated ABCD pseudo-concept maps.

For this pilot:

- **HAM10000** is sampled using diagnostic labels where available.
- **ISIC 2018 segmentation data** is sampled using lesion morphology because it does not contain diagnostic labels.
- Metrics are computed between XAI maps and pseudo-concept maps.
- Visual grids are generated for manual inspection.

Recommended output name:

```text
05_xai_pseudo_concept_evaluation_pilot.ipynb
```

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Optional, used only if available
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Change these paths if your repo uses different names
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs" / "phase5_pilot"
FIG_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Manifests created in earlier phases
HAM_MANIFEST = DATA_DIR / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_MANIFEST = DATA_DIR / "preprocessed_manifests" / "isic2018_preprocessed.csv"

# Map directories from earlier phases. Adjust if needed.
XAI_DIR = ROOT / "outputs" / "xai_maps"
CONCEPT_DIR = ROOT / "outputs" / "pseudo_concept_maps"

# Expected XAI methods and pseudo-concepts
XAI_METHODS = ["gradcam", "lime", "shap"]
CONCEPTS = ["asymmetry", "border_irregularity", "colour_variation"]

# Pilot size
N_HAM = 50
N_ISIC = 50
N_TOTAL = N_HAM + N_ISIC

# Evaluation settings
TOP_K_PERCENT = 20  # top 20% saliency pixels
EPS = 1e-8

print("ROOT:", ROOT)
print("Output directory:", OUTPUT_DIR)

## 2. Helper functions

These helpers keep the notebook robust if some files or maps are missing.

In [ ]:
def read_image(path: Path, size=(224, 224), grayscale=False) -> np.ndarray:
    """Read an image or map and resize to 224 x 224."""
    mode = "L" if grayscale else "RGB"
    img = Image.open(path).convert(mode)
    resample = Image.Resampling.NEAREST if grayscale else Image.Resampling.BILINEAR
    img = img.resize(size, resample=resample)
    arr = np.asarray(img)
    if grayscale:
        arr = arr.astype(np.float32) / 255.0
    return arr


def normalize_map(x: np.ndarray) -> np.ndarray:
    """Normalize a map to [0, 1]."""
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    xmin, xmax = float(x.min()), float(x.max())
    if xmax - xmin < EPS:
        return np.zeros_like(x, dtype=np.float32)
    return (x - xmin) / (xmax - xmin)


def topk_binary(x: np.ndarray, top_k_percent: float = 20) -> np.ndarray:
    """Return binary mask for the top-k percent values in a map."""
    x = normalize_map(x)
    threshold = np.percentile(x, 100 - top_k_percent)
    return (x >= threshold).astype(np.uint8)


def binary_from_map(x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    """Convert normalized map to binary mask."""
    return (normalize_map(x) >= threshold).astype(np.uint8)


def dice_score(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool)
    b = b.astype(bool)
    denom = a.sum() + b.sum()
    if denom == 0:
        return np.nan
    return 2 * np.logical_and(a, b).sum() / denom


def iou_score(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool)
    b = b.astype(bool)
    union = np.logical_or(a, b).sum()
    if union == 0:
        return np.nan
    return np.logical_and(a, b).sum() / union


def saliency_inside_ratio(saliency: np.ndarray, region: np.ndarray) -> float:
    """Fraction of saliency mass inside a region."""
    saliency = normalize_map(saliency)
    region = region.astype(bool)
    total = saliency.sum()
    if total < EPS:
        return np.nan
    return saliency[region].sum() / total


def mean_inside_outside_ratio(saliency: np.ndarray, region: np.ndarray) -> float:
    """Mean saliency inside concept region divided by mean saliency outside."""
    saliency = normalize_map(saliency)
    region = region.astype(bool)
    if region.sum() == 0 or (~region).sum() == 0:
        return np.nan
    inside = saliency[region].mean()
    outside = saliency[~region].mean()
    return inside / (outside + EPS)


def pearson_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = normalize_map(a).ravel()
    b = normalize_map(b).ravel()
    if a.std() < EPS or b.std() < EPS:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def find_map_file(base_dir: Path, dataset: str, method_or_concept: str, stem: str):
    """Try common filename layouts and return first existing file."""
    candidates = [
        base_dir / dataset / method_or_concept / f"{stem}.npy",
        base_dir / dataset / method_or_concept / f"{stem}.npz",
        base_dir / dataset / method_or_concept / f"{stem}.png",
        base_dir / method_or_concept / dataset / f"{stem}.npy",
        base_dir / method_or_concept / dataset / f"{stem}.npz",
        base_dir / method_or_concept / dataset / f"{stem}.png",
        base_dir / dataset / f"{stem}_{method_or_concept}.npy",
        base_dir / dataset / f"{stem}_{method_or_concept}.npz",
        base_dir / dataset / f"{stem}_{method_or_concept}.png",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def load_map_file(path: Path) -> np.ndarray:
    """Load .npy, .npz, or .png map."""
    if path.suffix.lower() == ".npy":
        return normalize_map(np.load(path))
    if path.suffix.lower() == ".npz":
        data = np.load(path)
        key = data.files[0]
        return normalize_map(data[key])
    return normalize_map(read_image(path, grayscale=True))

## 3. Load manifests

The notebook expects the preprocessed manifests from earlier phases.

If your local CSVs use different column names, update the column mapping in the next cell.

In [ ]:
ham = pd.read_csv(HAM_MANIFEST) if HAM_MANIFEST.exists() else pd.DataFrame()
isic = pd.read_csv(ISIC_MANIFEST) if ISIC_MANIFEST.exists() else pd.DataFrame()

print("HAM rows:", len(ham))
print("ISIC rows:", len(isic))
print("HAM columns:", list(ham.columns)[:20])
print("ISIC columns:", list(isic.columns)[:20])

## 4. Create the 100-image pilot sample

HAM is sampled using diagnosis labels.

ISIC is sampled using mask morphology because diagnostic labels are not available.

In [ ]:
def sample_ham_pilot(df: pd.DataFrame, n_total: int = 50) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["dataset"] = "HAM"

    # Expected columns from earlier pipeline: stem, dx, label, label_name.
    # If label is missing, derive melanoma/non-melanoma from dx == 'mel'.
    if "label" not in df.columns and "dx" in df.columns:
        df["label"] = (df["dx"] == "mel").astype(int)
    if "label_name" not in df.columns and "label" in df.columns:
        df["label_name"] = np.where(df["label"] == 1, "melanoma", "non_melanoma")

    melanoma = df[df.get("label", 0) == 1]
    non_melanoma = df[df.get("label", 0) == 0]

    n_mel = min(20, len(melanoma))
    n_non = min(20, len(non_melanoma))

    parts = []
    if n_mel:
        parts.append(melanoma.sample(n=n_mel, random_state=RANDOM_STATE))
    if n_non:
        parts.append(non_melanoma.sample(n=n_non, random_state=RANDOM_STATE))

    selected = pd.concat(parts, ignore_index=False) if parts else pd.DataFrame()

    remaining_n = n_total - len(selected)
    if remaining_n > 0:
        remaining = df.drop(index=selected.index, errors="ignore")
        # Prefer unusual cases: very small/large masks or border-touching if available.
        sort_cols = []
        if "touches_any_border" in remaining.columns:
            remaining["_priority_border"] = remaining["touches_any_border"].astype(int)
            sort_cols.append("_priority_border")
        if "mask_coverage" in remaining.columns:
            remaining["_priority_extreme_coverage"] = (remaining["mask_coverage"] - remaining["mask_coverage"].median()).abs()
            sort_cols.append("_priority_extreme_coverage")
        if sort_cols:
            remaining = remaining.sort_values(sort_cols, ascending=False)
            parts.append(remaining.head(remaining_n))
        else:
            parts.append(remaining.sample(n=min(remaining_n, len(remaining)), random_state=RANDOM_STATE))

    out = pd.concat(parts, ignore_index=True).head(n_total)
    out["pilot_group"] = "HAM_label_balanced"
    return out


def sample_isic_pilot(df: pd.DataFrame, n_total: int = 50) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["dataset"] = "ISIC"

    parts = []

    if "mask_size_class" in df.columns:
        for cls, n in [("tiny", 5), ("small", 10), ("normal", 15), ("large", 10), ("huge", 5)]:
            subset = df[df["mask_size_class"] == cls]
            if len(subset):
                parts.append(subset.sample(n=min(n, len(subset)), random_state=RANDOM_STATE))
    elif "mask_coverage" in df.columns:
        df["coverage_bin"] = pd.qcut(df["mask_coverage"], q=4, duplicates="drop")
        for _, subset in df.groupby("coverage_bin", observed=True):
            parts.append(subset.sample(n=min(10, len(subset)), random_state=RANDOM_STATE))

    selected = pd.concat(parts, ignore_index=False).drop_duplicates() if parts else pd.DataFrame()

    remaining_n = n_total - len(selected)
    if remaining_n > 0:
        remaining = df.drop(index=selected.index, errors="ignore")
        if "touches_any_border" in remaining.columns:
            border = remaining[remaining["touches_any_border"] == True]
            if len(border):
                add = border.sample(n=min(remaining_n, len(border)), random_state=RANDOM_STATE)
                selected = pd.concat([selected, add], ignore_index=False)
                remaining_n = n_total - len(selected)
        if remaining_n > 0:
            remaining = df.drop(index=selected.index, errors="ignore")
            selected = pd.concat([
                selected,
                remaining.sample(n=min(remaining_n, len(remaining)), random_state=RANDOM_STATE)
            ], ignore_index=False)

    out = selected.head(n_total).reset_index(drop=True)
    out["pilot_group"] = "ISIC_morphology_balanced"
    return out

ham_pilot = sample_ham_pilot(ham, N_HAM)
isic_pilot = sample_isic_pilot(isic, N_ISIC)

pilot = pd.concat([ham_pilot, isic_pilot], ignore_index=True)
pilot_path = METRICS_DIR / "phase5_pilot_100_sample.csv"
pilot.to_csv(pilot_path, index=False)

print("Pilot sample size:", len(pilot))
print(pilot[[c for c in ["dataset", "stem", "dx", "label", "label_name", "mask_coverage", "mask_size_class", "touches_any_border"] if c in pilot.columns]].head())
print("Saved:", pilot_path)

## 5. Resolve image, mask, XAI, and pseudo-concept paths

This section creates columns pointing to available files.

You may need to adjust the path rules if your local output folders use different names.

In [ ]:
def first_existing_path(row, candidates):
    for col in candidates:
        if col in row and pd.notna(row[col]):
            p = Path(str(row[col]))
            if p.exists():
                return p
            # Try relative to ROOT
            p2 = ROOT / str(row[col])
            if p2.exists():
                return p2
    return None

# Common manifest columns for image/mask paths
IMAGE_COL_CANDIDATES = ["image_path", "img_path", "preprocessed_image_path", "path"]
MASK_COL_CANDIDATES = ["mask_path", "preprocessed_mask_path"]

resolved_rows = []
for _, row in pilot.iterrows():
    row = row.copy()
    dataset = row["dataset"]
    stem = str(row["stem"])

    row["resolved_image_path"] = first_existing_path(row, IMAGE_COL_CANDIDATES)
    row["resolved_mask_path"] = first_existing_path(row, MASK_COL_CANDIDATES)

    for method in XAI_METHODS:
        row[f"xai_{method}_path"] = find_map_file(XAI_DIR, dataset.lower(), method, stem)

    for concept in CONCEPTS:
        row[f"concept_{concept}_path"] = find_map_file(CONCEPT_DIR, dataset.lower(), concept, stem)

    resolved_rows.append(row)

pilot_resolved = pd.DataFrame(resolved_rows)
resolved_path = METRICS_DIR / "phase5_pilot_100_resolved_paths.csv"
pilot_resolved.to_csv(resolved_path, index=False)

print("Saved:", resolved_path)

# Coverage report
path_cols = [c for c in pilot_resolved.columns if c.endswith("_path")]
coverage = {c: pilot_resolved[c].notna().sum() for c in path_cols}
pd.Series(coverage).sort_values(ascending=False)

## 6. Compute pilot metrics

For each image, XAI method, and pseudo-concept, this computes:

- IoU between top-k XAI pixels and concept region
- Dice between top-k XAI pixels and concept region
- Saliency Inside Ratio
- Mean saliency inside/outside concept ratio
- Pearson correlation between continuous XAI map and concept map

In [ ]:
records = []

for _, row in tqdm(pilot_resolved.iterrows(), total=len(pilot_resolved)):
    dataset = row["dataset"]
    stem = str(row["stem"])

    for method in XAI_METHODS:
        xai_path = row.get(f"xai_{method}_path")
        if xai_path is None or pd.isna(xai_path):
            continue
        try:
            xai_map = load_map_file(Path(xai_path))
            xai_bin = topk_binary(xai_map, TOP_K_PERCENT)
        except Exception as e:
            print(f"Could not load XAI map {xai_path}: {e}")
            continue

        for concept in CONCEPTS:
            concept_path = row.get(f"concept_{concept}_path")
            if concept_path is None or pd.isna(concept_path):
                continue
            try:
                concept_map = load_map_file(Path(concept_path))
                concept_bin = binary_from_map(concept_map, threshold=0.5)
            except Exception as e:
                print(f"Could not load concept map {concept_path}: {e}")
                continue

            records.append({
                "dataset": dataset,
                "stem": stem,
                "xai_method": method,
                "concept": concept,
                "top_k_percent": TOP_K_PERCENT,
                "iou": iou_score(xai_bin, concept_bin),
                "dice": dice_score(xai_bin, concept_bin),
                "sir": saliency_inside_ratio(xai_map, concept_bin),
                "inside_outside_ratio": mean_inside_outside_ratio(xai_map, concept_bin),
                "pearson_corr": pearson_corr(xai_map, concept_map),
                "concept_area_ratio": concept_bin.mean(),
                "xai_area_ratio": xai_bin.mean(),
            })

metrics = pd.DataFrame(records)
metrics_path = METRICS_DIR / "phase5_pilot_100_xai_concept_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print("Metric rows:", len(metrics))
print("Saved:", metrics_path)
metrics.head()

## 7. Summarise pilot metrics

In [ ]:
if len(metrics):
    summary = (
        metrics
        .groupby(["dataset", "xai_method", "concept"], dropna=False)
        .agg(
            n=("stem", "nunique"),
            mean_iou=("iou", "mean"),
            mean_dice=("dice", "mean"),
            mean_sir=("sir", "mean"),
            mean_inside_outside_ratio=("inside_outside_ratio", "mean"),
            mean_pearson_corr=("pearson_corr", "mean"),
        )
        .reset_index()
        .sort_values(["dataset", "xai_method", "concept"])
    )
else:
    summary = pd.DataFrame()

summary_path = METRICS_DIR / "phase5_pilot_100_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)
summary

## 8. Visual sanity-check grids

This generates example figures for manual inspection.

Each row shows:

```text
image | mask | XAI map | asymmetry | border irregularity | colour variation
```

If some maps are missing, the corresponding panel is blank.

In [ ]:
def show_or_blank(ax, arr, title, cmap=None):
    ax.axis("off")
    ax.set_title(title, fontsize=9)
    if arr is None:
        ax.text(0.5, 0.5, "missing", ha="center", va="center")
        return
    if cmap:
        ax.imshow(arr, cmap=cmap)
    else:
        ax.imshow(arr)


def get_image_from_row(row):
    p = row.get("resolved_image_path")
    if p is not None and pd.notna(p) and Path(p).exists():
        return read_image(Path(p), grayscale=False)
    return None


def get_mask_from_row(row):
    p = row.get("resolved_mask_path")
    if p is not None and pd.notna(p) and Path(p).exists():
        return read_image(Path(p), grayscale=True)
    return None


def make_visual_grid(rows: pd.DataFrame, method="gradcam", max_images=12, save_path=None):
    rows = rows.head(max_images)
    n = len(rows)
    ncols = 3 + len(CONCEPTS)
    fig, axes = plt.subplots(n, ncols, figsize=(3 * ncols, 3 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, (_, row) in enumerate(rows.iterrows()):
        stem = str(row["stem"])
        image = get_image_from_row(row)
        mask = get_mask_from_row(row)

        xai = None
        xai_path = row.get(f"xai_{method}_path")
        if xai_path is not None and pd.notna(xai_path) and Path(xai_path).exists():
            xai = load_map_file(Path(xai_path))

        show_or_blank(axes[r, 0], image, f"{row['dataset']}\n{stem}")
        show_or_blank(axes[r, 1], mask, "mask", cmap="gray")
        show_or_blank(axes[r, 2], xai, method, cmap="hot")

        for i, concept in enumerate(CONCEPTS):
            concept_map = None
            concept_path = row.get(f"concept_{concept}_path")
            if concept_path is not None and pd.notna(concept_path) and Path(concept_path).exists():
                concept_map = load_map_file(Path(concept_path))
            show_or_blank(axes[r, 3 + i], concept_map, concept, cmap="viridis")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# Generate one grid per dataset for Grad-CAM first
for dataset in ["HAM", "ISIC"]:
    subset = pilot_resolved[pilot_resolved["dataset"] == dataset]
    save_path = FIG_DIR / f"phase5_pilot_{dataset.lower()}_gradcam_grid.png"
    make_visual_grid(subset, method="gradcam", max_images=10, save_path=save_path)
    print("Saved:", save_path)

## 9. Optional: inspect best and worst alignments

This helps detect whether high scores visually make sense and whether low scores indicate actual disagreement or pipeline problems.

In [ ]:
if len(metrics):
    for metric_name in ["dice", "sir", "pearson_corr"]:
        print("\n", "=" * 80)
        print("Metric:", metric_name)
        display(metrics.sort_values(metric_name, ascending=False).head(10))
        display(metrics.sort_values(metric_name, ascending=True).head(10))

## 10. Notes for the full Phase 5 run

After this pilot, check:

1. Are all expected map files being resolved correctly?
2. Are visual overlays plausible?
3. Are any concept maps nearly empty or covering the whole lesion?
4. Are metrics stable across HAM and ISIC?
5. Should the top-k threshold be changed from 20% to 10% or 30%?

If the pilot is valid, reuse the same metric functions for the full dataset.